In [3]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

try:
    sh = gc.open("Sweet Crumbs Orders")
except gspread.exceptions.SpreadsheetNotFound:
    sh = gc.create("Sweet Crumbs Orders")

ws = sh.sheet1
try:
    # Only write headers if the sheet is empty to avoid duplicates on multiple runs
    if not ws.get_all_values():
        ws.append_row(["name", "item", "qty", "date", "eggless", "total"])
except Exception:
    ws.append_row(["name", "item", "qty", "date", "eggless", "total"])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1KhvUBbLyoYJxaZCGVzDPXQ1cZxoxaRWt40VYi2_kKYk


In [4]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
         "sunday": "10 am to 6 pm"}

class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    eggless: bool = Field(description="True if the customer asked for eggless")

In [5]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date, eggless=eggless)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date,
                   "yes" if order.eggless else "no", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

In [6]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for Sweet Crumbs Home Bakery.
FACTS:  Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # keep the whole run as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("Are you open on Sunday? How much are 2 chocolate cakes?"))
print(chat("I'm Ravi. Please book 2 eggless chocolate cakes for 24 Dec."))
print(chat(""))

Yes, we’re open Sunday from 10 am to 6 pm. Two chocolate cakes cost Rs 1300. Let me know your name and pickup date to place the order.
✅ Order confirmed!  
2 eggless chocolate cakes for Ravi, pickup on 24 Dec. Total: Rs 1300.  
We’ll see you then—thanks for choosing Sweet Crumbs!
Sure thing! If you need anything else—extra decorations, a note, or have any questions—just let me know. Happy baking!


### 1. Test Pydantic Class with `with_structured_output` on 5 Messy Messages

In [12]:
# Bind the structured output schema to the LLM
structured_llm = llm.with_structured_output(Order)

messy_messages = [
    "hey can i get a choclate cake for Sriharsha this sunday, make sure it is eggless",
    "Plum cake 1 unit please. Name is Dhanush. For dec 25th",
    "i want 10 cupcakes for my daughter's birthday party tomorrow. I am Prasad.",
    "Hey this is Hemanth. I need a chocolate cake for 28th Dec, eggless if possible please!",
    "Can you book 3 plum cakes for Raghavendra on New Year's Eve?"
]

for i, msg in enumerate(messy_messages, 1):
    print(f"--- Message {i}: '{msg}' ---")
    try:
        structured_order = structured_llm.invoke(msg)
        print(structured_order)
    except Exception as e:
        print("Extraction failed:", e)
    print()

--- Message 1: 'hey can i get a choclate cake for Sriharsha this sunday, make sure it is eggless' ---
name='Sriharsha' item='chocolate cake' qty=1 date='this sunday' eggless=True

--- Message 2: 'Plum cake 1 unit please. Name is Dhanush. For dec 25th' ---
name='Dhanush' item='plum cake' qty=1 date='dec 25th' eggless=False

--- Message 3: 'i want 10 cupcakes for my daughter's birthday party tomorrow. I am Prasad.' ---
name='Prasad' item='cupcake' qty=10 date='tomorrow' eggless=False

--- Message 4: 'Hey this is Hemanth. I need a chocolate cake for 28th Dec, eggless if possible please!' ---
name='Hemanth' item='chocolate cake' qty=1 date='28th Dec' eggless=True

--- Message 5: 'Can you book 3 plum cakes for Raghavendra on New Year's Eve?' ---
name='Raghavendra' item='plum cake' qty=3 date='2024-12-31' eggless=False



### 2. Break It: Rename `get_price` to `tool1` with docstring "does stuff"

In [13]:
@tool
def tool1(item: str, qty: int = 1) -> str:
    """Mangled tool that does stuff"""
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

# Create a broken agent using tool1 instead of get_price
broken_agent = create_agent(model=llm, tools=[tool1, is_open, log_order], system_prompt=SYSTEM)

try:
    response = broken_agent.invoke({"messages": [{"role": "user", "content": "How much are 5 cupcakes?"}]})
    print("Mangled Tool Agent Response:")
    for msg in response["messages"]:
        msg.pretty_print()
except Exception as e:
    print("Agent broke:", e)

Mangled Tool Agent Response:
================================ Human Message =================================

How much are 5 cupcakes?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_ca792bce-0370-4f4b-b9b5-0432e6282405)
 Call ID: fc_ca792bce-0370-4f4b-b9b5-0432e6282405
  Args:
    day: sunday
================================= Tool Message =================================
Name: is_open

Open 10 am to 6 pm on sunday.
================================== Ai Message ==================================
Tool Calls:
  tool1 (fc_d589c663-bc49-42fa-8395-54ac0a9d9589)
 Call ID: fc_d589c663-bc49-42fa-8395-54ac0a9d9589
  Args:
    item: cupcakes
================================= Tool Message =================================
Name: tool1

'cupcakes' is not on the menu. We have: plum cake, chocolate cake, cupcake.
================================== Ai Message ==================================

Sure! Could you let me know the pickup date and

### 3. Read the Trace: Pretty Print messages and count calls

In [14]:
# We will invoke the main agent and count tool vs model calls
result = agent.invoke({"messages": [{"role": "user", "content": "Are you open on Wednesday? Also log an order for 1 plum cake for Vinay on Monday, eggless."}]})

print("--- AGENT TRACE (pretty_print) ---\n")
tool_calls_count = 0
model_calls_count = 0

for msg in result["messages"]:
    msg.pretty_print()
    # Count model generations
    if msg.type == "ai":
        model_calls_count += 1
        # Check if model made tool calls
        if msg.tool_calls:
            tool_calls_count += len(msg.tool_calls)

print("\n--- TRACE SUMMARY ---")
print(f"Total model generations: {model_calls_count}")
print(f"Total tool invocations: {tool_calls_count}")

--- AGENT TRACE (pretty_print) ---

================================ Human Message =================================

Are you open on Wednesday? Also log an order for 1 plum cake for Vinay on Monday, eggless.
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_9d87f78b-d996-4027-98f0-368fdb060af4)
 Call ID: fc_9d87f78b-d996-4027-98f0-368fdb060af4
  Args:
    day: wednesday
================================= Tool Message =================================
Name: is_open

Open 9 am to 8 pm on wednesday.
================================== Ai Message ==================================
Tool Calls:
  log_order (fc_c4744966-8746-4493-82eb-1000b040bacd)
 Call ID: fc_c4744966-8746-4493-82eb-1000b040bacd
  Args:
    date: Monday
    eggless: True
    item: plum cake
    name: Vinay
    qty: 1
================================= Tool Message =================================
Name: log_order

Saved: 1 x plum cake for Vinay on Monday, Rs 500.
======